In [11]:
import pandas as pd
from MarketDepthTnS import validate_ob_data, validate_tns_data, run_all_experiments, plot_experiment, plot_summary
import os
import ast
os.chdir(r"D:\Study\Programs\trading")

In [2]:
date_ = "29APR2026"
security = "NIFTY2650524200CE.xlsx"

path_ = f"assets/logs/{date_}/extracted_symbols/{security}"
tns_path = f"assets/logs/{date_}/tns/{security}"
ob_df = pd.read_excel(path_)
ob_df['depth'] = ob_df['depth'].apply(ast.literal_eval)
tns_df = pd.read_excel(tns_path)

### Renamed Cols

In [3]:
import pandas as pd

def flatten_depth(row):
    depth = row['depth']
    data = {}

    # Process Bids (Buy side)
    for i in range(5):
        # i+1 to match your 1-5 naming convention
        data[f'bid_p{i+1}'] = depth['buy'][i]['price']
        data[f'bid_q{i+1}'] = depth['buy'][i]['quantity']

    # Process Asks (Sell side)
    for i in range(5):
        data[f'ask_p{i+1}'] = depth['sell'][i]['price']
        data[f'ask_q{i+1}'] = depth['sell'][i]['quantity']

    return pd.Series(data)

# Apply the function to create your new columns
depth_df = ob_df.apply(flatten_depth, axis=1)

# Join it back to your original dataframe (optional)
ob_df = pd.concat([ob_df, depth_df], axis=1)

# Rename specific columns using a dictionary mapping
ob_df.rename(columns={
    'last_trade_time': 'timestamp',
    'total_buy_quantity': 'buy_limit_qty',
    'total_sell_quantity': 'sell_limit_qty'
}, inplace=True)

In [4]:
ob_df.shape

(31963, 42)

In [5]:
ob_df = validate_ob_data(ob_df)

[OK] Order book data: 31962 valid ticks loaded.


In [6]:
ob_df.shape

(31962, 42)

In [7]:
# Rename specific columns using a dictionary mapping
tns_df.rename(columns={
    'time': 'timestamp',
    'ltp': 'price'
}, inplace=True)

In [8]:
tns_df = validate_tns_data(tns_df)

[OK] T&S data: 24614 valid prints loaded.


In [9]:
results = run_all_experiments(ob_df, tns_df)


  MARKET SENTIMENT EXPERIMENTS
[OK] Order book data: 31962 valid ticks loaded.
[OK] T&S data: 24614 valid prints loaded.
[OK] Base features computed. 31962 rows have valid forward labels.
     Forward window: 20s | Move threshold: 0.5

Running Experiment 1: Rolling OBI Trend...
  Accuracy: 50.48%  |  Signals: 8166  |  Baseline: 51.81%
Running Experiment 2: OBI + Mid Confluence...
  Accuracy: 49.34%  |  Signals: 3944  |  Baseline: 52.46%
Running Experiment 3: Absorption Detection...
  [WARN] Too few signal+outcome pairs to evaluate (<10). Adjust thresholds or use more data.
Running Experiment 5: OBI Divergence...
  Accuracy: 52.0%  |  Signals: 4777  |  Baseline: 51.08%
Running Experiment 4: T&S Aggression Proxy...
  Accuracy: 49.26%  |  Signals: 10673  |  Baseline: 51.64%

  RESULTS SUMMARY
                Experiment  Accuracy %  Baseline %  Edge %  Signals Verdict
         Rolling OBI Trend       50.48       51.81   -1.33     8166   NOISE
OBI + Mid-Price Confluence       49.34       5

In [15]:
plot_experiment(results, exp_number=2)

In [16]:
plot_summary(results)

In [14]:
pwd

'D:\\Study\\Programs\\trading'